## Feed-Forward Network

$$\text{FFN}(x) = \text{GELU}(xW_1 + b_1)W_2 + b_2$$

### 形状

| テンソル | 形状 |
|---|---|
| $x$ | $(\text{seq}, d_{model})$ |
| $W_1$ | $(d_{model}, 4d_{model})$ |
| $W_2$ | $(4d_{model}, d_{model})$ |

### 流れ

$$x \xrightarrow{W_1} \text{GELU} \xrightarrow{W_2} \text{out}$$

- $W_1$: $d_{model}$ を 4 倍に拡大し、より豊かな特徴を表現する
- $W_2$: 元の $d_{model}$ に戻す


In [ ]:
import torch
import torch.nn as nn


In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):  # d_ff=4×d_model　dropout=0.1は学習中にランダムに一部のニューロンの出力を 0 にする正則化。0.1 は「各要素を 10% の確率で 0 にする」割合。目的は過学習を防ぐこと。特定のニューロンに依存しすぎないようにする
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()
    
    def forward(self, x): # x: (batch, seq, d_model)
        x = self.linear1(x)  # → (batch, seq_len, d_ff)
        x = self.activation(x)  # GELU活性化
        x = self.dropout(x)
        x = self.linear2(x)  # → (batch, seq_len, d_model)
        return x


In [3]:
#テスト
ffn = PositionwiseFeedForward(d_model=512, d_ff=2048)
x = torch.randn(2, 10, 512)  #マスの数 = 2 × 10 × 512 = 10,240 個 その一つ一つに平均0, 分散1の乱数を入れる
output = ffn(x)

print("入力形状:", x.shape)
print("出力形状:", output.shape)
print("パラメータ数:", sum(p.numel() for p in ffn.parameters()))


# ffn.parameters() … モデル内の全パラメータ(linear1 の重みとバイアス、linear2 の重みとバイアス)を1つずつ取り出す
# p.numel() … そのパラメータの要素数(number of elements)を返す
# sum(... for p in ...) … 全部足し合わせる

入力形状: torch.Size([2, 10, 512])
出力形状: torch.Size([2, 10, 512])
パラメータ数: 2099712
